# 🛰️ Sentinel-2 Download Pipeline Notebook: Overview
## **Automated Spatial Querying → Multithreaded Data Retrieval**

This notebook serves as the **core data acquisition engine** for the Sentinel-2 download workflow. It automates the complex process of querying, filtering, and downloading satellite imagery from the Copernicus Data Space Ecosystem (CDSE) in a single, robust pipeline. It is designed to take a defined Region of Interest (ROI) and temporal constraints, securely authenticate with the Keycloak service, and execute highly efficient parallel downloads to prepare raw data for downstream processing.

---

### 🚀 Core Functional Areas
* **Spatial Validation & Verification**: Utilises interactive `folium` mapping to visually verify the Region of Interest (ROI) and auto-corrects coordinate reprojections prior to initiating large-scale API queries.
* **Automated Data Orchestration**: Queries the Copernicus OData catalogue using strict temporal windows, product type filters, and an intelligent overlap reduction algorithm to discard redundant or heavily clouded tiles.
* **Multithreaded Retrieval & Provisioning**: Accelerates data acquisition via parallel downloading workflows, followed by an automated batch extraction process that unpacks compressed `.zip` archives into analysis-ready `.SAFE` structures.

---

### 📋 Execution Requirements & Troubleshooting
* **CDSE Credentials**: This pipeline requires active user credentials for the Copernicus Data Space Ecosystem to generate OAuth2 bearer tokens. You must register an account if you do not have one.
* **Storage Allocation**: Sentinel-2 `.SAFE` archives are exceptionally large (often >1GB per scene). Ensure you have sufficient local or network drive capacity before initiating large temporal queries.
* **Overlap Reduction Tuning**: If no tiles are downloading, your `min_roi_coverage_frac` might be too high, or your `cloud_max` too strict. Adjust these parameters to cast a wider net over your study area.

---

### 👤 Author: Julian Manning
♻️ Last updated: **April 2026**  
📝 Project: **Sentinel-2 Download Workflow**  
📧 [julian.manning@outlook.com](mailto:julian.manning@outlook.com)  
🔗 Connect on [LinkedIn](https://www.linkedin.com/in/julian-manning/)  

<p style="text-align:left">
    <a href="https://www.esa.int/Applications/Observing_the_Earth/Copernicus/Sentinel-2" target="_blank">
        <img src="https://sentiwiki.copernicus.eu/__attachments/1671710/image-20230517-132224.png?inst-v=c933ac4b-944a-4344-ade1-8006f46193ba" width="400" alt="Graphic">
    </a>
</p>

## ⚙️ Workspace Configuration & Dependency Imports
> ### **Standard Library Initialisation → Custom Module Integration**

This involves configuring the system path to recognise the local `Tools` directory and importing the core orchestration functions from the **Sentinel-2 Download Pipeline Workflow** utility.

---

### 📂 Orchestration Components
| Function | Responsibility |
| :--- | :--- |
| **`visualise_roi_on_map`** | Generates an interactive Folium map visualization of the Region of Interest (ROI) for spatial verification. |
| **`get_roi_wkt_from_vector`** | Reads a vector dataset, handles coordinate reprojection, and extracts its bounding box as a WKT polygon string. |
| **`download_copernicus_data`** | Orchestrates the end-to-end pipeline for querying, filtering, and downloading Copernicus satellite data. |
| **`download_timer`** | Context manager utility that measures, formats, and logs the execution runtime of a code block. |

In [1]:
# Core
import sys
import getpass
from pathlib import Path

# Geospatial
import geopandas as gpd

# Ensure the 'Tools' directory is in the path for custom module discovery
sys.path.insert(0, "Tools")

from s2_download_helper_utility import (
    visualise_roi_on_map,
    get_roi_wkt_from_vector,
    download_copernicus_data,
    download_timer,
    batch_extract_zips
)

## 🗺️ Region of Interest (ROI) Initialisation & Spatial Validation
> ### **Vector Data Ingestion → Interactive Cartographic Verification**

This section defines the **Region of Interest (ROI)** and performs a quick **spatial sanity check** to confirm geographic alignment. The ROI geometry is overlaid on an interactive basemap using `folium`, allowing for fast visual inspection to ensure the study area is located where expected before any analysis is run.

---

### 📂 Logic & Orchestration Components
| Component | Responsibility | Analytical Objective |
| :--- | :--- | :--- |
| **`ROI_SHP_PATH`** | **Universal Path Resolution** | Utilises `Path` objects to ensure cross-platform compatibility for local shapefile directories. |
| **`ROI_GDF`** | **Vector Data Ingestion** | Reads the shapefile directly into a GeoDataFrame for geometric processing. |
| **`visualize_roi_on_map`** | **Spatial Sanity Check** | Renders an interactive map to verify ROI alignment against global basemaps. |

---

### 📋 Optional Tips
* **Basemap Customisation**: Supports multiple visual styles via the `tiles` argument. Common options include:
    * `"OpenStreetMap"` (default, reliable)
    * `"CartoDB positron"` or `"CartoDB positronNoLabels"` (clean, publication‑friendly)
    * `"CartoDB voyager"` (slightly higher contrast)
    * `"Stamen TonerLite"` (minimal, black‑and‑white)
    * `"Esri.WorldImagery"` (satellite imagery)
    * `"CartoDB dark_matter"` (dark background for presentations)
* **Geometric Aggregation**: Set `use_dissolve=True` if your ROI consists of multiple features.
* **Reporting & Export**: Use `save_path=` to export the map as a standalone HTML file for reporting or sharing.

In [2]:
# 1. Define your path
ROI_SHP_PATH = Path(r"Example\SHP\Study_Area_EPSG_32756.shp")
# 2. Visualise it (optional sanity check)
ROI_GDF = gpd.read_file(ROI_SHP_PATH)
display(visualise_roi_on_map(ROI_GDF, tiles="CartoDB positron"))
# 3. Extract the WKT 
roi_wkt_string = get_roi_wkt_from_vector(ROI_SHP_PATH)

Reprojecting ROI from WGS 84 / UTM zone 56S to WGS 84...


## 🔐 Authentication & Credential Management
> ### **Secure Runtime Input → CDSE Keycloak Identity Verification**

This handles the secure, hidden ingestion of user credentials at runtime, which are required to request an OAuth2 bearer token from the Copernicus Data Space Ecosystem (CDSE) Keycloak service.

> 🌐 **Prerequisite**: Sign into your Copernicus account as you normally do [here](https://dataspace.copernicus.eu/).  
> ⚠️ **Note**: If you do not have an active CDSE account, you must register a new profile before proceeding.

---

### 📂 Authentication Parameters
| Variable | Responsibility |
| :--- | :--- |
| **`copernicus_user`** | Dynamically captures and stores your registered CDSE email/username via a secure text input prompt. |
| **`copernicus_password`** | Masked runtime input that securely handles your account password to prevent cleartext exposure in logs. |

In [3]:
# copernicus User email
copernicus_user = getpass.getpass("Enter your email address")
# copernicus User Password
copernicus_password = getpass.getpass("Enter your password")

Enter your email address ········
Enter your password ········


## 🚀 Pipeline Execution & Parameter Configuration
> ### **Pipeline Parameterization → Threaded Orchestration & Profiling**

This stage configures the temporal, architectural, and quality constraints for the dataset before spinning up a multithreaded download environment wrapped in an execution profile timer.

---

### 📂 Configuration Parameters
| Parameter | Default/Value | Responsibility |
| :--- | :--- | :--- |
| **`time_begin` / `time_end`** | `'2026-01-01'` / `'2026-01-15'` | Constrains the Copernicus OData query strictly to this temporal window (Format: `YYYY-MM-DD`). |
| **`product_type`** | `["L1A", "L1B", "L1C"]` | Filters out scenes that do not match these specific Sentinel-2 processing levels (e.g., `["L1C"]`, `["L2A"]`, or `["L1C", "L2A"]`). |
| **`save_folder`** | `Path(r"DATA")` | Target directory path where raw, compressed satellite imagery is streamed. |
| **`download_latest_only`** | `True` | Deduplicates scenes by retaining only the highest/newest processing baseline version. |
| **`enable_overlap_reduction`** | `True` | Activates spatial thresholding to drop tiles with negligible intersection area (skips tiles that barely touch the ROI). |
| **`min_roi_coverage_frac`** | `0.70` | Mandates that a tile must cover at least 70% of the ROI to be flagged for download. |
| **`cloud_max`** | `20.0` | Discards any satellite tiles exceeding a 20% aggregate cloud cover threshold. |
| **`max_parallel_downloads`** | `3` | Spins up a `ThreadPoolExecutor` using 3 parallel worker threads for concurrent, accelerated I/O. |
| **`time_out`** | `2400` | Hard threshold limit (in seconds) for HTTP requests before dropping dead connections. |

---

### 🗺️ Sentinel-2 Naming Convention
The pipeline dynamically parses product filenames based on the standard Sentinel-2 SAFE format pattern:
`S2X_MSILX_YYYYMMDDTHHMMSS_NXXXX_RXXX_TXXXXX_YYYYMMDDTHHMMSS.SAFE`

* **`S2X`**: Satellite identifier (`S2A`, `S2B`, or `S2C`).
* **`MSILX`**: Processing level (`L1C` for Top-of-Atmosphere, `L2A` for Bottom-of-Atmosphere).
* **`YYYYMMDDTHHMMSS`**: Sensing start date and time.
* **`NXXXX`**: **Baseline number**. A larger numeric index indicates a more up-to-date processing version.
* **`RXXX`**: Relative orbit number.
* **`TXXXXX`**: **Tile identifier** corresponding to the MGRS grid square.
* **`YYYYMMDDTHHMMSS`**: Product generation date and time.
* **`.SAFE`**: Standard Archive Format for Europe suffix.

> 📌 **Version Selection Mechanics**: When multiple processing baselines exist for the identical sensing window and tile, the system evaluates the baseline token:
> * `..._20210520T235251_N0500_...SAFE` (Newer Baseline ✅)
> * `..._20210520T235251_N0300_...SAFE` (Older Baseline ❌)
> 
> Because **`download_latest_only=True`**, the architecture purges the outdated reprocessed footprint and strictly targets the `N0500` asset. If toggled to `False`, both files are marked for download.


In [5]:
# Set your parameters here
time_begin = '2026-01-01'
time_end = '2026-01-15'
product_type = ["L1A", "L1B", "L1C"]
save_folder = Path(r"DATA")

# Run the end-to-end download pipeline and track the time
with download_timer("Sentinel-2 Download Pipeline"):
    try:
        downloaded_files = download_copernicus_data(
            data_collection="SENTINEL-2",
            ROI=roi_wkt_string,
            time_begin=time_begin,                 # Format: YYYY-MM-DD
            time_end=time_end,                     # Format: YYYY-MM-DD
            copernicus_user=copernicus_user,
            copernicus_password=copernicus_password,
            product_type=product_type,             # e.g., valid types are ["L1C"], ["L2A"], ["L1C", "L2A"]
            save_folder=str(save_folder),          # Converts Path object to string for the function
            time_out=2400,                         # Timeout in seconds
            download_latest_only=True,             # Keeps only the newest processing baseline
            enable_overlap_reduction=True,         # Filters out tiles that barely touch the ROI
            max_parallel_downloads=3,              # Set >1 for faster threaded downloads
            min_roi_coverage_frac=0.70,            # Require at least 70% ROI coverage per tile
            cloud_max=20.0                        # Maximum cloud cover percentage
        )
        
        print(f"\nSuccessfully downloaded {len(downloaded_files)} files to: {save_folder}")
        
    except Exception as e:
        print(f"\x1b[91mPipeline failed during execution: {e}\x1b[0m")


--- Starting Sentinel-2 Download Pipeline ---
Start time: 2026-06-09 14:06:06

Starting query to Copernicus Data Space...
Query URL:
https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name%20eq%20'SENTINEL-2'%20and%20OData.CSC.Intersects(area=geography'SRID=4326;POLYGON%20((151.2491373659107%20-23.874256552989852,%20151.2491373659107%20-23.668065396256353,%20151.0710072390928%20-23.668065396256353,%20151.0710072390928%20-23.874256552989852,%20151.2491373659107%20-23.874256552989852))')%20and%20ContentDate/Start%20gt%202026-01-01T00:00:00.000Z%20and%20ContentDate/Start%20lt%202026-01-15T00:00:00.000Z%20and%20Attributes/OData.CSC.DoubleAttribute/any(att:att/Name%20eq%20'cloudCover'%20and%20att/OData.CSC.DoubleAttribute/Value%20le%2020.0)&$count=True&$top=1000
Raw product count: 4
All available tiles:
S2A_MSIL1C_20260103T001251_N0511_R073_T56KLU_20260103T012735.SAFE
S2B_MSIL1C_20260106T001109_N0511_R073_T56KKU_20260106T024936.SAFE
S2B_MSIL1C_20260106T001109_N0